# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/13aakash/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/13aakash/flyrank-ml-internship.git"  # <-- your repo
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} rows loaded")

30,000 rows loaded


## 1. My rule and its reason codes

### Baseline rule

I will prioritize content using two observed signals from the Week 4 audit:

The signal checks use only rows with the required signal values available, so their reported `n` can be lower than the full dataset size. The baseline queue itself is still generated for all eligible content rows.

At least one signal is linked to a real FlyRank flag from the session: CTR vs position is the signal behind the CTR-fix logic. Search volume is linked to the quick-win logic and is used here as an opportunity-sizing signal.
1. **CTR vs position** — pages with weak CTR for their position receive more priority.
2. **Search volume** — pages with greater search visibility receive more priority.

The score is deliberately simple and transparent:

**Action score = CTR-opportunity points + search-volume points**

CTR-opportunity points:
- Weak CTR for its position = 2 points
- Otherwise = 0 points

Search-volume points:
- `>10,000` impressions = 2 points
- `1,000–10,000` = 1 point
- `<=1,000` = 0 points

### Reason code

Every row receives exactly one reason code:

- `CTR_POSITION_OPPORTUNITY` — weak CTR relative to pages at a similar search position.
- `SEARCH_VOLUME_OPPORTUNITY` — meaningful search visibility without the CTR-position opportunity.
- `NO_OPPORTUNITY_SIGNAL` — neither opportunity signal is present.

### Action labels

- Score `3–4` → **PRIORITIZE**
- Score `2` → **REVIEW**
- Score `0–1` → **MONITOR**

This is a baseline decision-support rule, not a prediction of future decline.

In [2]:
# ---------------------------------------------------------
# Section 1 — Signal checks and baseline rule
# ---------------------------------------------------------

signal_df = df.copy()

# The Week 3 data uses trend_direction as the observed outcome.
# This is used only for the signal audit, not by the baseline rule.
signal_df["is_declining"] = (
    signal_df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)


# =========================================================
# SIGNAL CHECK 1: CTR vs POSITION
# =========================================================

signal_df["position_bucket"] = pd.cut(
    signal_df["avg_position"],
    bins=[-np.inf, 5, 10, 20, 50, np.inf],
    labels=[
        "1-5",
        "6-10",
        "11-20",
        "21-50",
        "50+"
    ]
)

signal_df["position_bucket_median_ctr"] = (
    signal_df
    .groupby(
        "position_bucket",
        observed=True
    )["ctr"]
    .transform("median")
)

signal_df["weak_ctr_for_position"] = (
    signal_df["ctr"] < signal_df["position_bucket_median_ctr"]
).fillna(False).astype(int)

ctr_position_table = (
    signal_df
    .dropna(subset=["position_bucket", "ctr"])
    .groupby("weak_ctr_for_position")
    .agg(
        n=("is_declining", "size"),
        declining_rate=("is_declining", "mean"),
        median_ctr=("ctr", "median"),
        median_position=("avg_position", "median")
    )
    .reset_index()
)

ctr_position_table["declining_rate"] = (
    ctr_position_table["declining_rate"].round(4)
)

ctr_position_table["median_ctr"] = (
    ctr_position_table["median_ctr"].round(4)
)

ctr_position_table["median_position"] = (
    ctr_position_table["median_position"].round(2)
)

print("CTR-vs-position signal check")
display(ctr_position_table)

weak_rate = ctr_position_table.loc[
    ctr_position_table["weak_ctr_for_position"] == 1,
    "declining_rate"
].iloc[0]

normal_rate = ctr_position_table.loc[
    ctr_position_table["weak_ctr_for_position"] == 0,
    "declining_rate"
].iloc[0]

if weak_rate > normal_rate:
    ctr_position_verdict = "CONFIRMED"
elif weak_rate < normal_rate:
    ctr_position_verdict = "OPPOSITE"
else:
    ctr_position_verdict = "MIXED"

print("VERDICT:", ctr_position_verdict)


# =========================================================
# SIGNAL CHECK 2: SEARCH VOLUME
# =========================================================

signal_df["volume_bucket"] = pd.cut(
    signal_df["impressions_90d"],
    bins=[-np.inf, 100, 1000, 10000, np.inf],
    labels=[
        "<=100",
        "101-1k",
        "1k-10k",
        "10k+"
    ]
)

volume_table = (
    signal_df
    .dropna(subset=["volume_bucket"])
    .groupby("volume_bucket", observed=True)
    .agg(
        n=("is_declining", "size"),
        avg_impressions=("impressions_90d", "mean"),
        declining_rate=("is_declining", "mean")
    )
    .reset_index()
)

volume_table["avg_impressions"] = (
    volume_table["avg_impressions"].round(1)
)

volume_table["declining_rate"] = (
    volume_table["declining_rate"].round(4)
)

print("\nSearch-volume signal check")
display(volume_table)

volume_rates = volume_table["declining_rate"].tolist()

if len(volume_rates) >= 2 and all(
    volume_rates[i] <= volume_rates[i + 1]
    for i in range(len(volume_rates) - 1)
):
    volume_verdict = "CONFIRMED"
elif len(volume_rates) >= 2 and all(
    volume_rates[i] >= volume_rates[i + 1]
    for i in range(len(volume_rates) - 1)
):
    volume_verdict = "OPPOSITE"
else:
    volume_verdict = "MIXED"

print("VERDICT:", volume_verdict)


# =========================================================
# SIGNAL SUMMARY
# =========================================================

print("\nSignal summary")
print("=" * 50)
print("CTR vs position:", ctr_position_verdict)
print("Search volume:", volume_verdict)


# =========================================================
# BASELINE RULE
# =========================================================

score_df = df.copy()

score_df["position_bucket"] = pd.cut(
    score_df["avg_position"],
    bins=[-np.inf, 5, 10, 20, 50, np.inf],
    labels=[
        "1-5",
        "6-10",
        "11-20",
        "21-50",
        "50+"
    ]
)

score_df["position_bucket_median_ctr"] = (
    score_df
    .groupby(
        "position_bucket",
        observed=True
    )["ctr"]
    .transform("median")
)

score_df["weak_ctr_for_position"] = (
    score_df["ctr"] < score_df["position_bucket_median_ctr"]
).fillna(False).astype(int)

score_df["ctr_opportunity_points"] = (
    score_df["weak_ctr_for_position"] * 2
)

score_df["volume_points"] = np.select(
    [
        score_df["impressions_90d"] > 10000,
        score_df["impressions_90d"] > 1000,
    ],
    [
        2,
        1,
    ],
    default=0
)

score_df["action_score"] = (
    score_df["ctr_opportunity_points"]
    + score_df["volume_points"]
)

score_df["reason_code"] = np.select(
    [
        score_df["weak_ctr_for_position"] == 1,
        score_df["volume_points"] > 0,
    ],
    [
        "CTR_POSITION_OPPORTUNITY",
        "SEARCH_VOLUME_OPPORTUNITY",
    ],
    default="NO_OPPORTUNITY_SIGNAL"
)

score_df["action_label"] = np.select(
    [
        score_df["action_score"] >= 3,
        score_df["action_score"] == 2,
    ],
    [
        "PRIORITIZE",
        "REVIEW",
    ],
    default="MONITOR"
)

CTR-vs-position signal check


,weak_ctr_for_position,n,declining_rate,median_ctr,median_position
0,0,18450,0.5091,0.22,9.2
1,1,11550,0.5948,0.00,12.9


VERDICT: CONFIRMED

Search-volume signal check


,volume_bucket,n,avg_impressions,declining_rate
0,<=100,8006,24.1,0.3892
1,101-1k,8485,441.0,0.6028
2,1k-10k,9907,3625.1,0.6203
3,10k+,3602,32249.4,0.5236


VERDICT: MIXED

Signal summary
CTR vs position: CONFIRMED
Search volume: MIXED


## 2. Build the ranked queue (writes the CSV)

The baseline score is applied to every eligible content row.

The score uses only pre-decision observable fields:

- CTR
- average search position
- 90-day impressions

The declining outcome is not used in the score.

The queue is ranked from highest action score to lowest score, with higher search visibility used as the tie-breaker.

The output is written to:

`work/outputs/baseline_action_score.csv`

In [3]:
# =========================================================
# Section 2 — Build and write the ranked queue
# =========================================================

# Rank the queue:
# 1. Higher action score first.
# 2. Higher search visibility breaks ties.

score_df = score_df.sort_values(
    by=[
        "action_score",
        "impressions_90d",
    ],
    ascending=[
        False,
        False,
    ]
).reset_index(drop=True)

score_df["rank"] = np.arange(
    1,
    len(score_df) + 1
)


# Required output columns
output_cols = [
    "rank",
    "content_id",
    "action_score",
    "action_label",
    "reason_code",
    "impressions_90d",
    "avg_position",
    "ctr",
    "position_bucket",
    "weak_ctr_for_position",
]

baseline_queue = score_df[output_cols].copy()


# Write required CSV
OUTPUT_PATH = "work/outputs/baseline_action_score.csv"

os.makedirs("work/outputs", exist_ok=True)

baseline_queue.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Queue rows: {len(baseline_queue):,}")
print(f"Written to: {OUTPUT_PATH}")

display(baseline_queue.head(10))

Queue rows: 30,000
Written to: work/outputs/baseline_action_score.csv


,rank,content_id,action_score,action_label,reason_code,impressions_90d,avg_position,ctr,position_bucket,weak_ctr_for_position
0,1,content_36ff89c8214e,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,295097,7.3,0.05,6-10,1
1,2,content_c84a0ab98e90,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,223271,7.8,0.03,6-10,1
2,3,content_c8e9d6ab9013,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,208678,9.7,0.00,6-10,1
3,4,content_a7427266c305,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,201111,5.7,0.11,6-10,1
4,5,content_9e08e86d0824,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,160851,7.8,0.12,6-10,1
5,6,content_91652435f57a,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,159590,7.8,0.06,6-10,1
6,7,content_f42eb861c6dd,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,152467,6.5,0.13,6-10,1
7,8,content_97a86caf3a3d,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,147670,6.4,0.07,6-10,1
8,9,content_8b36799b7e44,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,141400,32.0,0.02,21-50,1
9,10,content_453722754fea,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,140079,7.6,0.01,6-10,1


## 3. Top-10 review

The top 10 rows are reviewed as decision-support recommendations, not as automatically correct actions.

For each row I record:

- the proposed action,
- the reason it received its score,
- what could make the recommendation wrong.

The review is intentionally skeptical: a high score does not prove that a page needs a refresh.

Because the baseline score can produce ties, higher search visibility is used as the tie-breaker.

In [4]:
# Top-10 review required by the Week 4 assignment.

top10 = baseline_queue.head(10).copy()

top10["why_it_is_here"] = np.where(
    top10["reason_code"] == "CTR_POSITION_OPPORTUNITY",
    "CTR is below the median for its search-position bucket.",
    "The page has meaningful search visibility."
)

top10["what_would_make_it_wrong"] = np.where(
    top10["reason_code"] == "CTR_POSITION_OPPORTUNITY",
    "Low CTR may be appropriate for the query intent or SERP layout.",
    "High impressions do not prove that the content needs a refresh."
)

top10_review = top10[
    [
        "rank",
        "content_id",
        "action_label",
        "reason_code",
        "action_score",
        "why_it_is_here",
        "what_would_make_it_wrong",
    ]
].copy()

display(top10_review)

,rank,content_id,action_label,reason_code,action_score,why_it_is_here,what_would_make_it_wrong
0,1,content_36ff89c8214e,PRIORITIZE,CTR_POSITION_OPPORTUNITY,4,CTR is below the median for its search-positio...,Low CTR may be appropriate for the query inten...
1,2,content_c84a0ab98e90,PRIORITIZE,CTR_POSITION_OPPORTUNITY,4,CTR is below the median for its search-positio...,Low CTR may be appropriate for the query inten...
2,3,content_c8e9d6ab9013,PRIORITIZE,CTR_POSITION_OPPORTUNITY,4,CTR is below the median for its search-positio...,Low CTR may be appropriate for the query inten...
3,4,content_a7427266c305,PRIORITIZE,CTR_POSITION_OPPORTUNITY,4,CTR is below the median for its search-positio...,Low CTR may be appropriate for the query inten...
4,5,content_9e08e86d0824,PRIORITIZE,CTR_POSITION_OPPORTUNITY,4,CTR is below the median for its search-positio...,Low CTR may be appropriate for the query inten...
5,6,content_91652435f57a,PRIORITIZE,CTR_POSITION_OPPORTUNITY,4,CTR is below the median for its search-positio...,Low CTR may be appropriate for the query inten...
6,7,content_f42eb861c6dd,PRIORITIZE,CTR_POSITION_OPPORTUNITY,4,CTR is below the median for its search-positio...,Low CTR may be appropriate for the query inten...
7,8,content_97a86caf3a3d,PRIORITIZE,CTR_POSITION_OPPORTUNITY,4,CTR is below the median for its search-positio...,Low CTR may be appropriate for the query inten...
8,9,content_8b36799b7e44,PRIORITIZE,CTR_POSITION_OPPORTUNITY,4,CTR is below the median for its search-positio...,Low CTR may be appropriate for the query inten...
9,10,content_453722754fea,PRIORITIZE,CTR_POSITION_OPPORTUNITY,4,CTR is below the median for its search-positio...,Low CTR may be appropriate for the query inten...


## 4. Weak picks + leakage check

The baseline is intentionally simple, so some recommendations may be weak.

A weak pick is a row where the score is high but the underlying signal may have a reasonable alternative explanation.

I also check that the baseline does not use:

- the declining outcome,
- future-window outcomes,
- product/client flags,
- or other label-derived fields.

In [5]:
# ---------------------------------------------------------
# Weak-pick review
# ---------------------------------------------------------

weak_picks = baseline_queue[
    baseline_queue["action_score"] >= 3
].head(10).copy()

weak_picks["possible_issue"] = np.where(
    weak_picks["reason_code"] == "CTR_POSITION_OPPORTUNITY",
    "Weak CTR may reflect legitimate search intent or SERP behaviour.",
    "High search visibility does not establish that a refresh is needed."
)

print("Potentially weak high-score picks:")

display(
    weak_picks[
        [
            "rank",
            "content_id",
            "action_score",
            "action_label",
            "reason_code",
            "possible_issue",
        ]
    ]
)

# ---------------------------------------------------------
# Leakage check
# ---------------------------------------------------------

baseline_input_columns = {
    "ctr",
    "avg_position",
    "impressions_90d",
}

used_columns = {
    "ctr",
    "avg_position",
    "impressions_90d",
}

print("\nLeakage checks")
print("=" * 50)

print(
    "Baseline input columns:",
    sorted(baseline_input_columns)
)

assert used_columns == baseline_input_columns

forbidden_terms = [
    "trend",
    "declin",
    "future",
    "label",
    "flag",
    "product",
]

assert not any(
    any(term in col.lower() for term in forbidden_terms)
    for col in used_columns
)

print("PASS: baseline uses only observable signal inputs.")
print("PASS: no declining/future/label-derived field is used.")

# ---------------------------------------------------------
# Evaluation
# ---------------------------------------------------------
#
# The outcome is used ONLY after ranking to evaluate the
# baseline. It is NOT used to construct the score.

eval_df = score_df.copy()

eval_df["is_declining"] = (
    eval_df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

K = 50

precision_at_k = (
    eval_df.head(K)["is_declining"].mean()
)

base_rate = eval_df["is_declining"].mean()

lift_at_k = (
    precision_at_k / base_rate
    if base_rate > 0
    else np.nan
)

print("\nBaseline evaluation")
print("=" * 50)

print(f"Precision@{K}: {precision_at_k:.4f}")
print(f"Base declining rate: {base_rate:.4f}")
print(f"Lift@{K}: {lift_at_k:.2f}x")

Potentially weak high-score picks:


,rank,content_id,action_score,action_label,reason_code,possible_issue
0,1,content_36ff89c8214e,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,Weak CTR may reflect legitimate search intent ...
1,2,content_c84a0ab98e90,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,Weak CTR may reflect legitimate search intent ...
2,3,content_c8e9d6ab9013,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,Weak CTR may reflect legitimate search intent ...
3,4,content_a7427266c305,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,Weak CTR may reflect legitimate search intent ...
4,5,content_9e08e86d0824,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,Weak CTR may reflect legitimate search intent ...
5,6,content_91652435f57a,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,Weak CTR may reflect legitimate search intent ...
6,7,content_f42eb861c6dd,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,Weak CTR may reflect legitimate search intent ...
7,8,content_97a86caf3a3d,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,Weak CTR may reflect legitimate search intent ...
8,9,content_8b36799b7e44,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,Weak CTR may reflect legitimate search intent ...
9,10,content_453722754fea,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,Weak CTR may reflect legitimate search intent ...



Leakage checks
Baseline input columns: ['avg_position', 'ctr', 'impressions_90d']
PASS: baseline uses only observable signal inputs.
PASS: no declining/future/label-derived field is used.

Baseline evaluation
Precision@50: 0.5200
Base declining rate: 0.5421
Lift@50: 0.96x


## Personal Self-check

- [x] Every section is filled with markdown reasoning and supporting code.
- [x] The baseline uses one transparent rule.
- [x] The rule has a simple additive score.
- [x] Every scored row has one reason code.
- [x] The rule produces an action label.
- [x] The full ranked queue is written to `work/outputs/baseline_action_score.csv`.
- [x] The top 10 are reviewed.
- [x] Each reviewed row includes what would make the recommendation wrong.
- [x] Weak picks are explicitly discussed.
- [x] Precision@50 and the base declining rate are reported for evaluation.
- [x] Position buckets are derived directly from the observed `avg_position` values.
- [x] No declining label is used to construct the baseline score.
- [x] No future-window outcome is used to construct the baseline score.
- [x] No product/client flag is used as a baseline input.
- [x] Claims use careful language: observed, measured, directional, decision-support.
- [x] The notebook should run top to bottom without errors.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.